# GPU Kernel Generation — Google Colab Runner

This notebook sets up the environment to run the `llm-mlir-compiler` pipeline on Google Colab.

**What it does:**
1. Sets the Gemini API key
2. Clones the repository
3. Installs Python dependencies
4. Extracts the MLIR Python bindings tarball
5. Runs the benchmark pipeline using Gemini for LLM inference

**Prerequisites:**
- A Google Colab account (free tier works)
- Turn on GPU acceleration: Runtime → Change runtime type → Hardware accelerator: T4
- A Gemini API key from [Google AI Studio](https://aistudio.google.com/app/apikey)

In [1]:
# Cell 1: Set Gemini API key
# Get your key from: https://aistudio.google.com/app/apikey

import os
os.environ["GEMINI_API_KEY"] = "your_gemini_api_key_here"

# Verify it's set
assert os.environ["GEMINI_API_KEY"] != "your_gemini_api_key_here", "Please replace with your actual Gemini API key!"
print("✅ Gemini API key configured.")

✅ Gemini API key configured.


In [12]:
%cd /content
# Now it's safe to delete
!rm -rf /content/gpu-kernel-generation

/content


In [13]:
# Cell 2: Clone the repository

import os

REPO_URL = "https://github.com/seradotcom/gpu-kernel-generation.git"  # Update if needed
REPO_DIR = "/content/gpu-kernel-generation"

if not os.path.exists(REPO_DIR):
    !git clone -b GaelChanges {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!ls -la

Cloning into '/content/gpu-kernel-generation'...
remote: Enumerating objects: 177, done.
remote: Counting objects: 100% (177/177), done.
remote: Compressing objects: 100% (114/114), done.
remote: Total 177 (delta 104), reused 134 (delta 61), pack-reused 0 (from 0)
Receiving objects: 100% (177/177), 129.29 KiB | 1.58 MiB/s, done.
Resolving deltas: 100% (104/104), done.
/content/gpu-kernel-generation
total 124
drwxr-xr-x 5 root root  4096 Jun  9 03:36 .
drwxr-xr-x 1 root root  4096 Jun  9 03:36 ..
-rw-r--r-- 1 root root   450 Jun  9 03:36 benchmark_prompts.json
-rw-r--r-- 1 root root  8069 Jun  9 03:36 colab_run.ipynb
drwxr-xr-x 2 root root  4096 Jun  9 03:36 core
-rw-r--r-- 1 root root   702 Jun  9 03:36 .env.example
drwxr-xr-x 8 root root  4096 Jun  9 03:36 .git
-rw-r--r-- 1 root root   146 Jun  9 03:36 .gitignore
-rw-r--r-- 1 root root  8071 Jun  9 03:36 INSTALL_MLIR.md
-rw-r--r-- 1 root root  7498 Jun  9 03:36 kaggle_run.ipynb
-rw-r--r-- 1 root root 10193 Jun  9 03:36 main.py
-rw-r--

In [4]:
# Cell 3: Install Python dependencies

!pip install -q -r requirements.txt
print("✅ Python dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 7.1 MB/s eta 0:00:00
✅ Python dependencies installed.


In [5]:
from google.colab import drive
drive.mount('/content/drive')
import sys
import glob
# Point to the already-extracted folder in your Drive
EXTRACT_DIR = '/content/drive/MyDrive/llvm-install'
mlir_core_paths = glob.glob(f'{EXTRACT_DIR}/**/mlir_core', recursive=True)
if mlir_core_paths:
    MLIR_CORE_PATH = mlir_core_paths[0]
    sys.path.append(MLIR_CORE_PATH)
    print(f"✅ Found mlir_core at: {MLIR_CORE_PATH}")
else:
    print(f"❌ No mlir_core found under {EXTRACT_DIR}")
    print("Make sure llvm-install/ is in your Google Drive root.")
# Test import
try:
    from mlir.ir import Context
    print("✅ MLIR bindings import successful")
    MLIR_OK = True
except Exception as e:
    print(f"❌ MLIR import failed: {type(e).__name__}: {e}")
    print("\nThe extracted bindings may not be compatible with Colab's Python/glibc.")
    print("You can skip MLIR verification and rely on Pydantic + Semantic validation only.")
    MLIR_OK = False

Mounted at /content/drive
✅ Found mlir_core at: /content/drive/MyDrive/llvm-install/python_packages/mlir_core
✅ MLIR bindings import successful


In [6]:
import os
repo_path = "/content/gpu-kernel-generation"
print(f"Repo exists: {os.path.exists(repo_path)}")
print(f"core/ exists: {os.path.exists(os.path.join(repo_path, 'core'))}")
print(f"core/__init__.py exists: {os.path.exists(os.path.join(repo_path, 'core', '__init__.py'))}")
print(f"core/triton_python_generator.py exists: {os.path.exists(os.path.join(repo_path, 'core', 'triton_python_generator.py'))}")
print(f"core/triton_executor.py exists: {os.path.exists(os.path.join(repo_path, 'core', 'triton_executor.py'))}")

Repo exists: True
core/ exists: True
core/__init__.py exists: True
core/triton_python_generator.py exists: True
core/triton_executor.py exists: True


In [7]:
# Cell 5: Smoke test — verify all core modules import

import sys
sys.path.insert(0, "/content/gpu-kernel-generation")

from core.schemas import MlirResponse
from core.semantic_validator import SemanticValidator
from core.triton_python_generator import TritonPythonGenerator
from core.triton_executor import TritonExecutor
from core.prompt_builder import PromptBuilder

# MLIR translator depends on native bindings
if MLIR_OK:
    from core.mlir_translator import MLIRTranslator
    print("✅ MLIRTranslator imported")
else:
    print("⚠️  MLIRTranslator skipped (bindings not available)")

print("✅ All core modules imported successfully.")

✅ MLIRTranslator imported
✅ All core modules imported successfully.


In [69]:
# Cell 6: Run the benchmark pipeline (uses Gemini API)

%cd /content/gpu-kernel-generation
!python run_benchmarks.py

/content/gpu-kernel-generation
Traceback (most recent call last):
  File "/content/gpu-kernel-generation/run_benchmarks.py", line 7, in <module>
  File "/content/gpu-kernel-generation/core/llm_client.py", line 5, in <module>
    from google import genai
  File "/usr/local/lib/python3.12/dist-packages/google/genai/__init__.py", line 19, in <module>
    from . import types
  File "/usr/local/lib/python3.12/dist-packages/google/genai/types.py", line 10098, in <module>
    class GenerationConfig(_common.BaseModel):
  File "/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_model_construction.py", line 242, in __new__
    set_model_fields(cls, config_wrapper=config_wrapper, ns_resolver=ns_resolver)
  File "/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_model_construction.py", line 566, in set_model_fields
    fields, class_vars = collect_model_fields(cls, config_wrapper, ns_resolver, typevars_map=typevars_map)
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [14]:
!python run_dev_test_multi.py

[Info] Using MLIR bindings from Colab Drive (dynamic search): /content/drive/MyDrive/llvm-install/python_packages/mlir_core
=== MULTI-KERNEL DEV TEST ===
Testing pipeline on various kernel families without LLM calls.


KERNEL: vector_add
[1/5] Parsing JSON...
    -> Pydantic PASSED
[2/5] Semantic validation...
    -> Semantic PASSED
[3/5] MLIR translation...
    -> MLIR PASSED
[4/5] Triton Python generation...
    -> Triton Python PASSED

--- Generated Triton Python ---
import triton
import triton.language as tl
import torch

@triton.jit
def vec_sum_kernel(arg0_A, arg1_B, arg2_C):
    pid = tl.program_id(0)
    block_start = pid * 256
    offsets = block_start + tl.arange(0, 256)
    val_A = tl.load(arg0_A + offsets)
    val_B = tl.load(arg1_B + offsets)
    val_C = val_A + val_B
    tl.store(arg2_C + offsets, val_C)

---
[5/5] Compile and execute...
    -> Execution FAILED
    -> Error: Triton kernel execution failed:
Traceback (most recent call last):
  File "/content/gpu-kernel-gene

In [38]:
# cell 6 alt ver
!python -c "import sys; sys.path.insert(0,'/content/gpu-kernel-generation'); exec(open('/content/gpu-kernel-generation/run_benchmarks.py').read())" 2>&1 | head -100

[Info] Using MLIR bindings from sys.path: /content/drive/MyDrive/llvm-install/python_packages/mlir_core
=== Starting TritonBench LLM-MLIR Evaluator with Triton Python Feedback Loop ===
[Info] Running without MLOps tracking: No API key configured. Use `wandb login` to log in.

[VECTOR_ADD] (Difficulty: easy)
  [Attempt 1/3] Generating JSON via LLM...
  -> Pipeline exception: maximum recursion depth exceeded
  [Attempt 2/3] Generating JSON via LLM...
  -> Pipeline exception: maximum recursion depth exceeded
  [Attempt 3/3] Generating JSON via LLM...
  -> Pipeline exception: maximum recursion depth exceeded

[ELEMENT_MUL] (Difficulty: easy)
  [Attempt 1/3] Generating JSON via LLM...
  -> Pipeline exception: maximum recursion depth exceeded
  [Attempt 2/3] Generating JSON via LLM...
  -> Pipeline exception: maximum recursion depth exceeded
  [Attempt 3/3] Generating JSON via LLM...
  -> Pipeline exception: maximum recursion depth exceeded

=== Benchmark Summary ===
vector_add          : ex

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 7 (Optional): Test a single custom prompt interactively

import json
import sys
sys.path.insert(0, "/content/gpu-kernel-generation")

from core.llm_client import generate_llm_response
from core.schemas import MlirResponse
from core.semantic_validator import SemanticValidator
from core.triton_python_generator import TritonPythonGenerator
from core.triton_executor import TritonExecutor
from core.prompt_builder import PromptBuilder

if MLIR_OK:
    from core.mlir_translator import MLIRTranslator
    translator = MLIRTranslator()
else:
    translator = None

validator = SemanticValidator()
gen = TritonPythonGenerator()
executor = TritonExecutor()
prompt_builder = PromptBuilder()

user_prompt = "Generate a Triton kernel that performs element-wise addition of two float32 vectors."
system_prompt = prompt_builder.build_prompt(user_prompt, MlirResponse.model_json_schema())

print("Calling Gemini API...")
raw = generate_llm_response("gemini", system_prompt, user_prompt, schema=MlirResponse)
print("✅ Raw response received.")

# Clean markdown wrappers if present
clean = raw.strip()
if clean.startswith("```json"): clean = clean[7:]
if clean.startswith("```"): clean = clean[3:]
if clean.endswith("```"): clean = clean[:-3]
clean = clean.strip()

mlir_obj = MlirResponse(**json.loads(clean))
print("✅ Pydantic validation passed.")

# Semantic validation
errors = validator.validate(mlir_obj)
if errors:
    print(f"❌ Semantic errors: {errors}")
else:
    print("✅ Semantic validation passed.")

# MLIR translation (if available)
if translator:
    mlir_code = translator.translate_to_module(mlir_obj.code)
    print("✅ MLIR verification passed.")
else:
    print("⚠️  Skipping MLIR verification (bindings not available)")

# Generate Triton Python
triton_py = gen.generate(mlir_obj.code)
print("\n=== Generated Triton Python ===")
print(triton_py)

# Compile and benchmark
result = executor.run(triton_py, n_elements=1024)
print(f"\n=== Execution Result ===")
print(json.dumps(result, indent=2))

---

## Cell 8: THUNLP TritonBench Pipeline (New)

This runs the **THUNLP TritonBench** benchmark suite with our MLIR-verify-feedback loop.

Architecture:
1. **Phase 1**: Abstracted prompt → JSON MLIR → Semantic Validator → MLIRTranslator.verify()
2. **Phase 2**: Actual TritonBench prompt + MLIR feedback → unconstrained Triton Python
3. **Phase 3**: Evaluate generated kernel using THUNLP's native test harness

Results are saved to `output/tritonbench_pipeline_results.json`.

In [ ]:
# Cell 8: Run THUNLP TritonBench Pipeline

import sys
sys.path.insert(0, "/content/gpu-kernel-generation")

from run_tritonbench_pipeline import run_tritonbench_pipeline

# Run the pipeline on a small curated subset
results = run_tritonbench_pipeline()

print("\n=== Final Results ===")
import json
print(json.dumps(results, indent=2))


---

## Troubleshooting

**Git clone fails:** Colab sometimes blocks GitHub. Try uploading the repo as a zip file and extracting it instead.

**Gemini API rate limit:** If you get rate-limited, add `time.sleep(5)` between API calls or upgrade to a paid plan.

**MLIR bindings fail to import:** The tarball was compiled for a specific Linux environment. If it doesn't work on Colab, ask your teammate to re-compile inside a Colab notebook and replace the tarball.

**Triton compilation fails:** Make sure GPU is enabled (T4). Go to Runtime → Change runtime type → Hardware accelerator: T4.

**CUDA out of memory:** Reduce `n_elements` in `run_benchmarks.py` or the executor call.